# Day 9 · Exercise 1: Extract a Book Record

**What you'll build:** `extract_book(text: str, model: str) -> dict` — a function that sends a free-text book description to a local Ollama model and returns a validated dict with typed fields (title, author, year, genre, summary).

**Why it matters:** Schema-guided extraction is the pattern that converts unstructured prose into typed, validated data your code can sort, filter, and store — the first step in almost every LLM-powered data pipeline.

## Your Implementation

In [ ]:
import json
import ollama
from pydantic import BaseModel, Field


class BookRecord(BaseModel):
    title: str = Field(description="The book's title")
    author: str = Field(description="The author's full name")
    year: int = Field(description="Year of first publication as an integer")
    genre: str = Field(description="Primary literary genre")
    summary: str = Field(description="One-sentence plot summary")


def extract_book(text: str, model: str) -> dict:
    """Extract structured book information from a free-text description.

    Uses a Pydantic schema to tell the model exactly which fields to fill,
    then validates the JSON response so prose becomes typed, validated data.

    Args:
        text:  Free-text description or blurb of a book.
        model: Ollama model name to use (e.g. "llama3.2").

    Returns:
        A dict with keys: title (str), author (str), year (int),
        genre (str), summary (str).  Guaranteed by Pydantic validation.

    Example:
        extract_book(
            "George Orwell's Nineteen Eighty-Four, published in 1949, "
            "is a dystopian novel about Winston Smith's rebellion against "
            "the totalitarian Party.",
            model="llama3.2",
        )
        # -> {'title': 'Nineteen Eighty-Four', 'author': 'George Orwell',
        #     'year': 1949, 'genre': 'Dystopian fiction',
        #     'summary': '...'}
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

TEST_BLURB = (
    "George Orwell's Nineteen Eighty-Four, published in 1949, is a dystopian "
    "novel set in a totalitarian future society where independent thinking is a "
    "crime. The story follows Winston Smith as he secretly rebels against the "
    "all-seeing Party and its leader, Big Brother."
)

TEST_MODEL = "llama3.2"


def _run_checks():
    score, total = 0, 4

    # Check 1: function exists and is callable
    try:
        assert callable(extract_book), "extract_book is not defined or not callable"
        print(f"{_PASS} Check 1/{total}: extract_book is defined and callable")
        score += 1
    except Exception as e:
        print(f"{_FAIL} Check 1/{total}: {e}")
        return

    # Check 2: returns a dict
    try:
        result = extract_book(TEST_BLURB, TEST_MODEL)
        assert isinstance(result, dict), (
            f"expected dict, got {type(result).__name__}"
        )
        print(f"{_PASS} Check 2/{total}: extract_book returns a dict")
        score += 1
    except Exception as e:
        print(f"{_FAIL} Check 2/{total}: {e}")
        return

    # Check 3: all required keys are present
    try:
        required = {"title", "author", "year", "genre", "summary"}
        missing = required - result.keys()
        assert not missing, f"missing keys: {missing}"
        print(f"{_PASS} Check 3/{total}: dict contains all required keys (title, author, year, genre, summary)")
        score += 1
    except Exception as e:
        print(f"{_FAIL} Check 3/{total}: {e}")

    # Check 4: year is an int and the other core fields are non-empty strings
    try:
        assert isinstance(result.get("year"), int), (
            f"year must be int, got {type(result.get('year')).__name__}"
        )
        for key in ("title", "author", "genre", "summary"):
            val = result.get(key, "")
            assert isinstance(val, str) and val.strip(), (
                f"{key!r} must be a non-empty string, got {val!r}"
            )
        print(f"{_PASS} Check 4/{total}: year is int; title/author/genre/summary are non-empty strings")
        score += 1
    except Exception as e:
        print(f"{_FAIL} Check 4/{total}: {e}")

    print()
    if score == total:
        print("🎉 Exercise complete!")
    print(f"  {score}/{total} passed." + (" Keep going!" if score < total else ""))


_run_checks()

## Bonus Challenge

Right now `extract_book` raises a `ValidationError` if the model omits a field or returns `year` as a string it cannot coerce. In Lesson 2 (the next lesson today) you will learn about optional fields and graceful degradation.

As a preview: change `year` in `BookRecord` to `year: int | None = None`, re-run your function on a blurb that does not mention a publication date, and observe that it returns `None` for `year` instead of crashing. This is the pattern that makes extraction robust in production.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
import json
import ollama
from pydantic import BaseModel, Field


class BookRecord(BaseModel):
    title: str = Field(description="The book's title")
    author: str = Field(description="The author's full name")
    year: int = Field(description="Year of first publication as an integer")
    genre: str = Field(description="Primary literary genre")
    summary: str = Field(description="One-sentence plot summary")


def extract_book(text: str, model: str) -> dict:
    """Extract structured book information from a free-text description."""
    schema_json = json.dumps(BookRecord.model_json_schema(), indent=2)

    system_prompt = (
        "You are a data extraction assistant.\n"
        "The user will send you a text passage. Extract the requested information\n"
        "from that passage and return a JSON object that matches this schema:\n\n"
        f"{schema_json}\n\n"
        "Return ONLY the JSON object with values taken from the user's text.\n"
        "Do not return the schema itself. No prose, no explanation."
    )

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": text},
        ],
        format="json",
    )

    raw = response["message"]["content"]
    record = BookRecord.model_validate_json(raw)
    return record.model_dump()
```

**Why this works:** Embedding the JSON Schema in the system prompt gives the model an unambiguous contract for what fields to produce and what types they must be. Setting `format="json"` activates Ollama's grammar-constrained sampling, which guarantees the response is valid JSON before Pydantic even sees it. Calling `model_validate_json()` then enforces type correctness — if the model writes `year` as a string, Pydantic coerces it to `int` or raises a `ValidationError` immediately, so bad data never flows silently into the rest of your pipeline.
</details>